In [ ]:
"""
=========================================================================
  FINAL RESULTS
===========================================================================
       λ |   Test Acc |   Sparsity @T=20 |   Neg Scores |  Gate Mean
  ──────-+-──────────-+-────────────────-+-────────────-+-──────────
     0.5 |     59.19% |           86.75% |       89.40% |     0.1063
     1.5 |     59.17% |           93.38% |       94.79% |     0.0522
     4.0 |     58.69% |           96.99% |       97.77% |     0.0224
===========================================================================
"""

In [4]:
"""
  Self-Pruning Neural Network on CIFAR-10

  """

import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\n{'='*65}")
print(f"  Self-Pruning Neural Network - CIFAR-10")
print(f"{'='*65}")
print(f"  Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
print(f"  Mechanism: Temperature-annealed sigmoid gates (T: 1 → 20)")
print(f"  Pruning: L1 penalty on gate values (normalised)")
print(f"{'='*65}\n")


# ─────────────────────────────────────────────────────────────────────────────
# 1. PrunableLinear  —  temperature-annealed sigmoid gates
# ─────────────────────────────────────────────────────────────────────────────
class PrunableLinear(nn.Module):
    """
    Linear layer with per-weight learnable gates.

    gate(s, T) = sigmoid(s * T)

    At low T (early training): gates are soft → gradients flow freely,
    the classification loss can differentiate which weights matter.

    At high T (late training): sigmoid is near-binary → gates commit to
    0 or 1. Combined with L1 on gate values, this creates true sparsity.

    The temperature T is NOT a parameter — it's a schedule passed in
    at each forward call. gate_scores ARE parameters and are updated
    by the optimizer with gradient = ∂L/∂s = ∂L/∂gate * T*σ(sT)(1-σ(sT)).
    At high T this derivative is large for scores near 0, accelerating
    the bifurcation into 0 or 1.
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # gate_scores: zero-mean small noise so gates start at ~0.5
        # Classification gradient will IMMEDIATELY differentiate them:
        # useful weights → positive scores → gates → 1
        # useless weights → negative scores → gates → 0 (with L1 help)
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)
        # Small noise: gates start at σ(noise) ≈ 0.5 ± small amount
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.01)

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        # At T=1 (start): soft gates, gradients flow freely
        # At T=20 (end):  near-binary gates, true pruning
        gates          = torch.sigmoid(self.gate_scores * temperature)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self, temperature: float = 20.0) -> torch.Tensor:
        """Return gate values at a given temperature (detached)."""
        with torch.no_grad():
            s = self.gate_scores.detach()
            return torch.sigmoid(s * temperature)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}"


# ─────────────────────────────────────────────────────────────────────────────
# 2. Network
# ─────────────────────────────────────────────────────────────────────────────
class SelfPruningNet(nn.Module):
    """
    MLP for CIFAR-10 using PrunableLinear layers.

    Architecture (kept intentionally simple per task spec):
      3072 → 1024 → 512 → 256 → 128 → 10

    Smaller than v1/v2 on purpose:
      - Fewer total gates → sparsity % is more meaningful
      - Less redundancy → pruning creates a real trade-off
      - Still enough capacity for ~58-62% on CIFAR-10
    """

    def __init__(self, dropout: float = 0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(3072, 1024);  self.bn1 = nn.BatchNorm1d(1024)
        self.fc2 = PrunableLinear(1024,  512);  self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = PrunableLinear( 512,  256);  self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = PrunableLinear( 256,  128);  self.bn4 = nn.BatchNorm1d(128)
        self.fc5 = PrunableLinear( 128,   10)
        self.drop = nn.Dropout(dropout)
        self._temperature = 1.0     # updated externally each epoch

    @property
    def temperature(self): return self._temperature
    @temperature.setter
    def temperature(self, val): self._temperature = float(val)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = self._temperature
        x = self.flatten(x)
        x = self.drop(F.relu(self.bn1(self.fc1(x, T))))
        x = self.drop(F.relu(self.bn2(self.fc2(x, T))))
        x = self.drop(F.relu(self.bn3(self.fc3(x, T))))
        x = F.relu(self.bn4(self.fc4(x, T)))
        return self.fc5(x, T)

    def prunable_layers(self):
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def _n_gates(self) -> int:
        return sum(l.gate_scores.numel() for l in self.prunable_layers())

    def sparsity_loss(self) -> torch.Tensor:
        """
        Normalised L1 of gate values at current temperature.
        Returns mean gate value ∈ [0, 1].
        Minimising this drives gates toward 0 (pruned).
        """
        total = torch.tensor(0., device=next(self.parameters()).device)
        n = 0
        for layer in self.prunable_layers():
            gates = torch.sigmoid(layer.gate_scores * self._temperature)
            total = total + gates.sum()
            n    += gates.numel()
        return total / n    # mean gate value, in [0, 1]

    def overall_sparsity(self, threshold: float = 0.01,
                         eval_temp: float = 20.0) -> float:
        """
        Sparsity measured at high temperature (eval_temp=20) to simulate
        the final committed state. At T=20, sigmoid is near-binary:
        gate_score < 0  → gate ≈ 0
        gate_score > 0  → gate ≈ 1
        """
        pruned = total = 0
        for layer in self.prunable_layers():
            g = layer.get_gates(temperature=eval_temp)
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return pruned / total if total else 0.

    def score_based_sparsity(self) -> float:
        """Fraction of gate_scores that are negative (→ gate→0 at high T)."""
        neg = total = 0
        for layer in self.prunable_layers():
            s = layer.gate_scores.detach()
            neg   += (s < 0).sum().item()
            total += s.numel()
        return neg / total if total else 0.

    def all_gate_values(self, eval_temp: float = 20.0) -> np.ndarray:
        vals = [l.get_gates(eval_temp).cpu().numpy().ravel()
                for l in self.prunable_layers()]
        return np.concatenate(vals)

    def gate_score_params(self):
        for l in self.prunable_layers(): yield l.gate_scores

    def non_gate_params(self):
        gate_ids = {id(l.gate_scores) for l in self.prunable_layers()}
        for p in self.parameters():
            if id(p) not in gate_ids: yield p


# ─────────────────────────────────────────────────────────────────────────────
# 3. Temperature schedule
# ─────────────────────────────────────────────────────────────────────────────
def get_temperature(epoch: int, total_epochs: int,
                    T_start: float = 1.0, T_end: float = 20.0) -> float:
    """
    Exponential annealing from T_start to T_end.

    Early epochs: T≈1 → gates soft, classification gradient differentiates scores
    Late epochs:  T≈20 → gates near-binary, L1 penalty commits pruning decisions

    Uses exponential (not linear) so most of training happens at useful mid-temps.
    """
    frac = (epoch - 1) / max(total_epochs - 1, 1)
    return T_start * (T_end / T_start) ** frac


def get_lambda_warmup(lam: float, epoch: int, warmup: int) -> float:
    """Linearly ramp λ from 0 to target over warmup epochs."""
    return lam * min(1.0, epoch / max(warmup, 1))


# ─────────────────────────────────────────────────────────────────────────────
# 4. Data loaders
# ─────────────────────────────────────────────────────────────────────────────
def get_dataloaders(batch_size: int = 256):
    mean = (0.4914, 0.4822, 0.4465);  std = (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    root = "./data"
    train_ds = torchvision.datasets.CIFAR10(root, train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10(root, train=False, download=True, transform=test_tf)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(test_ds,  batch_size=512,        shuffle=False, num_workers=2, pin_memory=True))


# ─────────────────────────────────────────────────────────────────────────────
# 5. Train / Eval
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, opt_main, opt_gate, lam_eff, epoch, total_ep):
    model.train()
    tot_loss = cls_sum = sp_sum = 0.; correct = n = 0

    for bi, (imgs, lbls) in enumerate(loader):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

        opt_main.zero_grad(); opt_gate.zero_grad()

        logits   = model(imgs)
        cls_loss = F.cross_entropy(logits, lbls)
        sp_loss  = model.sparsity_loss()            # normalised mean gate ∈ [0,1]
        loss     = cls_loss + lam_eff * sp_loss

        loss.backward()
        # Clip ONLY main network gradients; let gate gradients flow freely
        nn.utils.clip_grad_norm_(list(model.non_gate_params()), max_norm=5.0)
        opt_main.step(); opt_gate.step()

        bs = lbls.size(0)
        tot_loss += loss.item() * bs;  cls_sum += cls_loss.item() * bs
        sp_sum   += sp_loss.item() * bs
        correct  += (logits.argmax(1) == lbls).sum().item();  n += bs

        if (bi + 1) % 50 == 0 or (bi + 1) == len(loader):
            print(f"  Ep{epoch:>2}/{total_ep} [{bi+1:>3}/{len(loader)}] "
                  f"loss={tot_loss/n:.4f} cls={cls_sum/n:.4f} "
                  f"sp={sp_sum/n:.4f} acc={100.*correct/n:.1f}%",
                  end="\r")
    print()
    return tot_loss/n, cls_sum/n, sp_sum/n, correct/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        correct += (model(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
    return correct / total


# ─────────────────────────────────────────────────────────────────────────────
# 6. Full training run
# ─────────────────────────────────────────────────────────────────────────────
def train_run(lam: float, epochs: int = 30, lr: float = 1e-3,
              gate_lr: float = 5e-3, warmup: int = 5,
              T_start: float = 1.0, T_end: float = 20.0,
              batch_size: int = 256):
    """
    One full training run.

    Two separate optimizers:
      opt_main — updates weights, biases, BN params (lr=1e-3)
      opt_gate — updates gate_scores only (lr=5e-3, 5× higher)

    Temperature rises from T_start to T_end over training.
    λ warms up from 0 to target over `warmup` epochs.
    """
    print(f"\n{'─'*65}")
    print(f"  λ={lam}  epochs={epochs}  lr={lr}  gate_lr={gate_lr}")
    print(f"  T: {T_start}→{T_end}  warmup={warmup}ep  bs={batch_size}")
    print(f"{'─'*65}")

    train_loader, test_loader = get_dataloaders(batch_size)
    model = SelfPruningNet(dropout=0.3).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    n_gates  = sum(l.gate_scores.numel() for l in model.prunable_layers())
    print(f"  Params: {n_params:,}  |  Gate params: {n_gates:,}")
    print(f"  Init: gate_scores~N(0,0.01) → gates≈0.5  (will differentiate quickly)\n")

    # Two separate optimizers — cleaner than param groups for very different LRs
    opt_main = optim.Adam(list(model.non_gate_params()),  lr=lr,      weight_decay=1e-4)
    opt_gate = optim.Adam(list(model.gate_score_params()),lr=gate_lr, weight_decay=0.)

    sched_main = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=epochs)

    hist = {"train_acc":[], "test_acc":[], "sparsity":[], "score_sparsity":[],
            "gate_mean":[], "temp":[], "lam_eff":[]}
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        T       = get_temperature(epoch, epochs, T_start, T_end)
        lam_eff = get_lambda_warmup(lam, epoch, warmup)
        model.temperature = T   # set on model so forward() uses it

        tr_loss, cls_l, sp_l, tr_acc = train_epoch(
            model, train_loader, opt_main, opt_gate, lam_eff, epoch, epochs)
        sched_main.step()

        test_acc   = evaluate(model, test_loader)
        sparsity   = model.overall_sparsity(eval_temp=T)    # at current T
        sp_final   = model.overall_sparsity(eval_temp=20.)  # projected final
        score_sp   = model.score_based_sparsity()           # gate_scores < 0
        gates      = model.all_gate_values(eval_temp=T)

        for k, v in zip(["train_acc","test_acc","sparsity","score_sparsity","gate_mean","temp","lam_eff"],
                        [tr_acc, test_acc, sp_final, score_sp,
                         float(gates.mean()), T, lam_eff]):
            hist[k].append(v)

        print(
            f"  ▶ Ep{epoch:>2}/{epochs}  "
            f"train={tr_acc*100:.1f}%  test={test_acc*100:.1f}%  "
            f"sparsity@T={sparsity*100:.1f}%  "
            f"sparsity@T20={sp_final*100:.1f}%  "
            f"neg_scores={score_sp*100:.1f}%  "
            f"gate_mean={gates.mean():.3f}  "
            f"T={T:.1f}  λ_eff={lam_eff:.3f}  "
            f"t={int(time.time()-t0)}s"
        )

    # ── Final stats ───────────────────────────────────────────────────────────
    model.temperature = 20.
    final_acc  = evaluate(model, test_loader)
    final_sp   = model.overall_sparsity(eval_temp=20.)
    score_sp   = model.score_based_sparsity()
    gates_fin  = model.all_gate_values(eval_temp=20.)

    print(f"\n  ✦ Final test accuracy         : {final_acc*100:.2f}%")
    print(f"  ✦ Sparsity @T=20 (gates<0.01) : {final_sp*100:.2f}%")
    print(f"  ✦ Gate scores < 0             : {score_sp*100:.2f}%  (→ pruned at convergence)")
    print(f"  ✦ Gate stats @T=20            : "
          f"min={gates_fin.min():.4f}  mean={gates_fin.mean():.4f}  "
          f"median={np.median(gates_fin):.4f}  max={gates_fin.max():.4f}")
    print(f"  ✦ Pct gates < 0.1             : {100*(gates_fin<0.1).mean():.1f}%")
    print(f"  ✦ Pct gates > 0.9             : {100*(gates_fin>0.9).mean():.1f}%")

    return {"lam": lam, "model": model, "test_acc": final_acc,
            "sparsity": final_sp, "score_sparsity": score_sp,
            "history": hist, "gates": gates_fin}


# ─────────────────────────────────────────────────────────────────────────────
# 7. Plotting
# ─────────────────────────────────────────────────────────────────────────────
def plot_all(results: list, best: dict, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)

    # ── A) Gate distribution ──────────────────────────────────────────────────
    gates = best["gates"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(
        f"Gate Distribution at T=20 — λ={best['lam']}  "
        f"(acc={best['test_acc']*100:.1f}%  sparsity={best['sparsity']*100:.1f}%)",
        fontsize=12, fontweight="bold")

    axes[0].hist(gates, bins=120, color="#2563eb", edgecolor="none")
    axes[0].axvline(0.01, color="red", lw=1.5, ls="--", label="Prune threshold 0.01")
    axes[0].set_title("Full distribution [0, 1]"); axes[0].set_xlabel("Gate value")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].hist(gates[gates < 0.05], bins=60, color="#dc2626", edgecolor="none")
    axes[1].set_title("Zoom: gates < 0.05  (zero-spike)")
    axes[1].set_xlabel("Gate value"); axes[1].grid(alpha=0.3)

    axes[2].hist(gates[gates > 0.5], bins=60, color="#16a34a", edgecolor="none")
    axes[2].set_title("Zoom: gates > 0.5  (surviving weights)")
    axes[2].set_xlabel("Gate value"); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "gate_distribution.png"), dpi=150); plt.close()

    # ── B) Trade-off ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    lams  = [r["lam"] for r in results]
    accs  = [r["test_acc"]*100  for r in results]
    spars = [r["sparsity"]*100  for r in results]
    score_sp = [r["score_sparsity"]*100 for r in results]

    ax.plot(lams, accs, "o-", color="#2563eb", lw=2, ms=8, label="Test Accuracy (%)")
    ax.set_xlabel("λ"); ax.set_ylabel("Test Accuracy (%)", color="#2563eb")
    ax.tick_params(axis="y", colors="#2563eb")
    ax2 = ax.twinx()
    ax2.plot(lams, spars,    "s--", color="#dc2626", lw=2, ms=8, label="Sparsity @T=20 (%)")
    ax2.plot(lams, score_sp, "^-.", color="#ea580c", lw=2, ms=8, label="Neg scores (%) ")
    ax2.set_ylabel("Sparsity (%)", color="#dc2626")
    ax2.tick_params(axis="y", colors="#dc2626")
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc="center left", fontsize=9)
    ax.set_title("Sparsity vs Accuracy Trade-off", fontweight="bold")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "tradeoff.png"), dpi=150); plt.close()

    # ── C) Training curves ────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("Training Dynamics for All λ Values", fontsize=13, fontweight="bold")
    pal = ["#2563eb", "#16a34a", "#dc2626"]

    for i, r in enumerate(results):
        c = pal[i]; lbl = f"λ={r['lam']}"
        ep = range(1, len(r["history"]["train_acc"]) + 1)
        axes[0,0].plot(ep, [a*100 for a in r["history"]["train_acc"]], c, lw=1.8, label=lbl)
        axes[0,1].plot(ep, [a*100 for a in r["history"]["test_acc"]],  c, lw=1.8, label=lbl)
        axes[0,2].plot(ep, [s*100 for s in r["history"]["sparsity"]],  c, lw=1.8, label=lbl)
        axes[1,0].plot(ep, [s*100 for s in r["history"]["score_sparsity"]], c, lw=1.8, label=lbl)
        axes[1,1].plot(ep, r["history"]["gate_mean"], c, lw=1.8, label=lbl)
        axes[1,2].plot(ep, r["history"]["temp"], "k--", lw=1, label="Temperature" if i==0 else "")

    titles = ["Train Accuracy", "Test Accuracy", "Sparsity @T=20 (%)",
              "Negative Scores (%)", "Gate Mean @current T", "Temperature T"]
    ylabels = ["%", "%", "%", "%", "Gate value", "T"]
    for ax, t, y in zip(axes.ravel(), titles, ylabels):
        ax.set_title(t); ax.set_xlabel("Epoch"); ax.set_ylabel(y)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150); plt.close()

    print(f"\n  Plots saved to {save_dir}/")


# ─────────────────────────────────────────────────────────────────────────────
# 8. Summary table
# ─────────────────────────────────────────────────────────────────────────────
def print_table(results):
    print(f"\n{'='*75}")
    print(f"  FINAL RESULTS")
    print(f"{'='*75}")
    print(f"  {'λ':>6} | {'Test Acc':>10} | {'Sparsity @T=20':>16} | {'Neg Scores':>12} | {'Gate Mean':>10}")
    print(f"  {'─'*6}-+-{'─'*10}-+-{'─'*16}-+-{'─'*12}-+-{'─'*10}")
    for r in results:
        print(f"  {r['lam']:>6} | "
              f"{r['test_acc']*100:>9.2f}% | "
              f"{r['sparsity']*100:>15.2f}% | "
              f"{r['score_sparsity']*100:>11.2f}% | "
              f"{r['gates'].mean():>10.4f}")
    print(f"{'='*75}")
    print("\n  Notes:")
    print("  • Sparsity @T=20: gates below 0.01 when evaluated at high temperature")
    print("  • Neg Scores   : fraction of gate_scores < 0 → become pruned at convergence")
    print("  • These two metrics tell the full story of pruning success\n")


# ─────────────────────────────────────────────────────────────────────────────
# 9. Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 30 epochs per run × 3 runs = ~15 min on T4
    # λ values: low / medium / high sparsity targets
    LAMBDAS    = [0.5, 1.5, 4.0]
    EPOCHS     = 30
    LR         = 1e-3
    GATE_LR    = 5e-3     # 5× higher for gate_scores
    WARMUP     = 5        # 5 epoch λ ramp
    T_START    = 1.0      # soft sigmoid at epoch 1
    T_END      = 20.0     # near-binary at epoch 30
    BATCH_SIZE = 256
    SAVE_DIR   = "./outputs_v3"

    print(f"  Running {len(LAMBDAS)} experiments × {EPOCHS} epochs each")
    print(f"  λ values: {LAMBDAS}")
    print(f"  Temperature annealing: {T_START} → {T_END} (exponential)")
    print(f"  Total training epochs: {len(LAMBDAS) * EPOCHS}\n")

    results = []
    for lam in LAMBDAS:
        r = train_run(lam=lam, epochs=EPOCHS, lr=LR, gate_lr=GATE_LR,
                      warmup=WARMUP, T_start=T_START, T_end=T_END,
                      batch_size=BATCH_SIZE)
        results.append(r)

    print_table(results)

    # Best = highest accuracy among those with >20% sparsity; else best acc
    pruned = [r for r in results if r["sparsity"] > 0.20]
    best   = max(pruned, key=lambda r: r["test_acc"]) if pruned else max(results, key=lambda r: r["test_acc"])
    print(f"  Best model for plot: λ={best['lam']}  acc={best['test_acc']*100:.1f}%  "
          f"sparsity={best['sparsity']*100:.1f}%")

    plot_all(results, best, SAVE_DIR)
    print("\n  Done!\n")


  Self-Pruning Neural Network - CIFAR-10
  Device : cuda
  GPU    : Tesla T4
  Mechanism: Temperature-annealed sigmoid gates (T: 1 → 20)
  Pruning: L1 penalty on gate values (normalised)

  Running 3 experiments × 30 epochs each
  λ values: [0.5, 1.5, 4.0]
  Temperature annealing: 1.0 → 20.0 (exponential)
  Total training epochs: 90


─────────────────────────────────────────────────────────────────
  λ=0.5  epochs=30  lr=0.001  gate_lr=0.005
  T: 1.0→20.0  warmup=5ep  bs=256
─────────────────────────────────────────────────────────────────
  Params: 7,676,042  |  Gate params: 3,835,136
  Init: gate_scores~N(0,0.01) → gates≈0.5  (will differentiate quickly)

  Ep 1/30 [196/196] loss=1.9297 cls=1.8799 sp=0.4980 acc=31.8%
  ▶ Ep 1/30  train=31.8%  test=40.6%  sparsity@T=0.0%  sparsity@T20=0.1%  neg_scores=64.3%  gate_mean=0.496  T=1.0  λ_eff=0.100  t=18s
  Ep 2/30 [196/196] loss=1.7940 cls=1.6957 sp=0.4916 acc=38.7%
  ▶ Ep 2/30  train=38.7%  test=45.0%  sparsity@T=0.0%  sparsity@T20=4.4